# Residual value model

## 1.- Definition of inputs and outputs of the model

El objetivo de este modelo es desarrollar un predictor del valor residual de vehículos nuevos. Para ello, se empleará un modelo previamente entrenado para la estimación del precio de vehículos usados. Este modelo ha sido entrenado con datos de vehículos fabricados entre los años 2000 y 2025, y su salida corresponde al precio estimado del vehículo en el año 2025.

**Variables de entrada:**

- **useful_life**: Vida útil del vehículo, interpretada como el plazo de financiación, es decir, la duración del contrato de renting, multiopción o leasing.  
- **mileage**: Kilometraje máximo permitido, expresado en millas, durante la vida útil del vehículo, equivalente a la duración del contrato.  
- **original_price**: Precio del vehículo, expresado en USD, en el momento de adquisición; es decir, el precio de un vehículo nuevo con dichas características a fecha de hoy (2025).  
- **make**: Marca del vehículo.  
- **model**: Modelo del vehículo.  
- **vehicle_type**: Tipo o categoría del vehículo.  
- **fuel_type**: Tipo de motorización.  
- **mpg_city**: Consumo de combustible en ciudad, expresado en millas por galón (MPG).  
- **mpg_highway**: Consumo de combustible en carretera, expresado en millas por galón (MPG).  
- **engine_displacement**: Cilindrada del motor.  
- **engine_cylinders**: Número de cilindros del motor.  
- **transmission_type**: Tipo de transmisión del vehículo.  
- **num_speeds**: Número de marchas o velocidades.  
- **drive_train**: Tipo de tracción del vehículo.  
- **interior_color**: Color del interior del vehículo, representado en codificación RGB.  
- **exterior_color**: Color del exterior del vehículo, representado en codificación RGB.  

**Salida esperada del modelo:**

El valor residual del vehículo transcurridos *useful_life* años, siempre que no se haya excedido el límite de *mileage*.


The objective of this model is to develop a predictor of the residual value of new vehicles. To achieve this, a previously trained model for estimating the price of used vehicles will be employed. This model was trained on data from vehicles manufactured between 2000 and 2025, and its output corresponds to the estimated vehicle price as of the year 2025.

**Input features:**

- **useful_life**: The useful life of the vehicle, interpreted as the financing term — i.e., the duration of the leasing, rent-to-own, or multi-option contract.  
- **mileage**: Maximum allowed mileage, expressed in miles, during the vehicle's useful life, equivalent to the contract duration.  
- **original_price**: Price of the vehicle in USD at the time of acquisition — i.e., the cost of a new vehicle with those specifications as of 2025.  
- **make**: Vehicle brand.  
- **model**: Vehicle model.  
- **vehicle_type**: Type or category of the vehicle.  
- **fuel_type**: Type of powertrain.  
- **mpg_city**: Fuel consumption in city driving, measured in miles per gallon (MPG).  
- **mpg_highway**: Fuel consumption in highway driving, measured in miles per gallon (MPG).  
- **engine_displacement**: Engine displacement.  
- **engine_cylinders**: Number of engine cylinders.  
- **transmission_type**: Vehicle transmission type.  
- **num_speeds**: Number of gears or speeds.  
- **drive_train**: Vehicle drivetrain type.  
- **interior_color**: Interior color of the vehicle, represented using RGB encoding.  
- **exterior_color**: Exterior color of the vehicle, represented using RGB encoding.  

**Model output:**

The residual value of the vehicle after *useful_life* years, provided the mileage does not exceed the specified *mileage* limit.

## 2.- Model Rationale

Si simplificamos las variables de entrada del modelo de valor residual como (*useful_life*, *mileage*, *original_price*, *vehicle_features*), donde *vehicle_features* representa el resto de características estáticas del vehículo, el primer paso consiste en utilizar el modelo de predicción del precio de vehículos usados para estimar cuánto costaría a día de hoy —es decir, en 2025— un vehículo con características *vehicle_features*, *useful_life* años de antigüedad y un kilometraje inferior a *mileage* millas.

El modelo de predicción del precio de vehículos usados también emplea como entrada el precio original del vehículo, es decir, el precio de adquisición en el momento de compra. Una buena aproximación de este valor consiste en descontar el precio actual *original_price* a valor pasado *useful_life* años, utilizando como tasa de descuento el índice de crecimiento anual de precios en el sector de vehículos nuevos. Si denotamos este índice por $i$, se puede estimar el precio original histórico, que llamaremos *former_original_price*, mediante la siguiente fórmula:

$$
former\_original\_price = \frac{original\_price}{(1+i)^{useful\_life}}
$$

Por ejemplo, **si a día de hoy un Audi A4 nuevo**, con motor de gasolina de 2L y 4 cilindros, consumo de 23 mpg en ciudad y 32 mpg en carretera, transmisión automática, tracción delantera, color exterior blanco y color interior marrón, **cuesta 50.000 USD**, entonces se estima que **hace 5 años** un Audi A4 nuevo con esas mismas características costaba aproximadamente:

$$
former\_original\_price = \frac{50.000\,\text{USD}}{(1+3\%)^{5\,\text{años}}} = 43.130\,\text{USD}
$$

asumiendo que $i = 3\%$.

Cabe destacar que, si se dispone de una serie histórica de índices de crecimiento anual para el sector, es posible obtener una estimación más precisa del precio original. No obstante, por simplicidad, asumiremos de aquí en adelante que $i = 3\%$ es constante. Además, este procedimiento es necesario ya que no existen garantías de que en 2020 estuviera disponible en el mercado un modelo de Audi A4 con exactamente las mismas características técnicas para obtener su precio de una base de datos.

Continuando con el ejemplo, se puede emplear el modelo de predicción de precio de vehículos usados para estimar el valor actual (en 2025) de un Audi A4 adquirido hace 5 años (en 2020), con menos de *mileage* millas. Las variables de entrada para el modelo serían:

```python
inputs = {
    "years": useful_life,
    "mileage": mileage,
    "original_price": former_original_price,
    "vehicle_features": vehicle_features
}

Asumamos que la salida del modelo predice que el valor actual es de *price* = 22.000 USD. Según la definición de valor residual, podemos afirmar que el valor residual de un Audi A4 de 2020 con 5 años de antigüedad y un kilometraje inferior a *mileage* millas —motor de gasolina de cilindrada de 2L y 4 cilindros, consumo de 23 mpg en ciudad y 32 mpg en carretera, transmisión automática, tracción delantera, color exterior blanco y color interior marrón— es de 22.000 USD.

Finalmente, se puede deducir fácilmente que, en 2030, cuando finalice el contrato de financiación (renting, leasing o multiopción) del Audi A4 adquirido hoy en 2025, dicho vehículo tendrá 5 años de antigüedad y, por contrato, no deberá haber superado *mileage* millas. En consecuencia, su valor será equivalente al valor actual (2025) de un Audi A4 con 5 años de antigüedad y un kilometraje de *mileage* millas, con las mismas características técnicas, es decir, 22.000 USD más el ajuste correspondiente por inflación. Asumiendo una tasa de inflación constante del 2% anual durante los próximos cinco años, el valor estimado en 2030 será:

$$
future\_value = 22.000\,\text{USD} \times (1+2\%)^{5\,\text{años}} = 24.289\,\text{USD}
$$

Esto equivale a afirmar que un Audi A4 nuevo que en 2025 cuesta 50.000 USD —con motor de gasolina de 2L y 4 cilindros, 23 mpg en ciudad, 32 mpg en carretera, transmisión automática, tracción delantera, color exterior blanco y color interior marrón— tendrá un **valor residual de 24.289 USD** al cabo de 5 años, siempre que el kilometraje se mantenga por debajo de *mileage* millas.

If we simplify the input variables of the residual value model as (*useful_life*, *mileage*, *original_price*, *vehicle_features*), where *vehicle_features* represents the remaining static characteristics of the vehicle, the first step is to use the used vehicle price prediction model to estimate the current price —that is, as of 2025— of a vehicle with characteristics *vehicle_features*, *useful_life* years of age, and mileage below *mileage* miles.

The used vehicle price prediction model also requires the original price of the vehicle as an input —that is, the acquisition price at the time of purchase. A good approximation of this value is to discount the current price *original_price* to its past value *useful_life* years ago, using as discount rate the annual price growth index for the new vehicle sector. If we denote this index as $i$, the historical original price, referred to as *former_original_price*, can be estimated using the following formula:

$$
former\_original\_price = \frac{original\_price}{(1+i)^{useful\_life}}
$$

For example, **if a brand-new Audi A4 today** —with a 2L gasoline engine, 4 cylinders, 23 mpg city, 32 mpg highway fuel consumption, automatic transmission, front-wheel drive, white exterior, and brown interior— **costs 50,000 USD**, then it is estimated that **five years ago** a new Audi A4 with the same characteristics cost approximately:

$$
former\_original\_price = \frac{50.000\,\text{USD}}{(1+3\%)^{5\,\text{years}}} = 43.130\,\text{USD}
$$

assuming $i = 3\%$.

It is worth noting that, if a historical series of annual price growth indices for the sector is available, a more accurate estimate of the original price can be obtained. However, for simplicity, we will assume from now on that $i = 3\%$ is constant. Furthermore, this procedure is necessary because there is no guarantee that in 2020 a new Audi A4 with exactly the same technical specifications was available in the market to retrieve its price from a database.

Continuing with the example, the used vehicle price prediction model can be used to estimate the current value (in 2025) of an Audi A4 purchased five years ago (in 2020), with less than *mileage* miles. The input variables for the model would be:

```python
inputs = {
    "years": useful_life,
    "mileage": mileage,
    "original_price": former_original_price,
    "vehicle_features": vehicle_features
}


Let us assume that the output of the model is that the current value is *price* = 22,000 USD. According to the definition of residual value, we can conclude that the residual value of a 2020 Audi A4 with 5 years of age and less than *mileage* miles —with a 2L gasoline engine, 4 cylinders, 23 mpg city and 32 mpg highway fuel consumption, automatic transmission, front-wheel drive, white exterior, and brown interior— is 22,000 USD.

It then follows that in 2030, when the financing contract (leasing, renting, or balloon option) for the Audi A4 purchased in 2025 ends, the vehicle will be 5 years old and contractually required to have covered less than *mileage* miles. Therefore, its value will match that of an Audi A4 in 2025 with 5 years of age and the same mileage and specifications —that is, 22,000 USD adjusted for inflation. Assuming a constant annual inflation rate of 2% over the next five years, the estimated value in 2030 is:

$$
future\_value = 22.000\,\text{USD} \times (1+2\%)^{5\,\text{years}} = 24.289\,\text{USD}
$$

In other words, a brand-new Audi A4 that costs 50,000 USD in 2025 —with a 2L gasoline engine, 4 cylinders, 23 mpg in the city, 32 mpg on the highway, automatic transmission, front-wheel drive, white exterior, and brown interior— will have a **residual value of 24,289 USD** after 5 years, provided that it remains under *mileage* miles.


## 3.- Model development

In [1]:
from xgboost import XGBRegressor
import pandas as pd

def used_cars_price_prediction_model(model_input):
    price_prediction_model = XGBRegressor() # Load the price_prediction_model
    price_prediction_model.load_model("xgboost_model.json")
    return price_prediction_model.predict(model_input)

def is_variable_value(variable, value):
    return variable == value

def get_RGB_component(component, RGB_code):
    if component == 'R':
        return RGB_code[0]
    if component == 'G':
        return RGB_code[1]
    if component == 'B':
        return RGB_code[2]

def get_make_te(make):
    make_te_dict = pd.read_excel('make_te_dictionary.xlsx')
    row = make_te_dict[make_te_dict['make'].str.lower() == make.lower()]
    return row['min'].values[0]

def get_model_te(model):
    model_te_dict = pd.read_excel('model_te_dictionary.xlsx')
    row = model_te_dict[model_te_dict['model'].str.lower() == model.lower()]
    if not row.empty:
        return row['min'].values[0]
    else:
        model_te_dict = pd.read_excel('aux_model_te_dictionary.xlsx')
        row = model_te_dict[model_te_dict['model'].str.lower() == model.lower()]
        return row['mean'].values[0]

def residual_value_prediction_model(model_input, i1 = 0.03, i2 = 0.02):
    model_input['former_original_price'] = model_input['original_price']/pow(1 + i1 ,model_input['useful_life']) # Discount the original price useful_life years back

    used_cars_price_prediction_model_input = pd.DataFrame()

    # Data preprocessing to apply the used_cars_price_prediction_model
    used_cars_price_prediction_model_input['years'] = model_input['useful_life']
    used_cars_price_prediction_model_input['original_price'] = model_input['former_original_price']
    used_cars_price_prediction_model_input[['mileage','mpg_city','mpg_highway']]=model_input[['mileage','mpg_city','mpg_highway']]
    used_cars_price_prediction_model_input['make_te'] = model_input['make'].apply(get_make_te)
    used_cars_price_prediction_model_input['model_te'] = model_input['model'].apply(get_model_te)
    used_cars_price_prediction_model_input[['num_speeds','engine_displacement','engine_cylinders']] = model_input[['num_speeds','engine_displacement','engine_cylinders']]

    used_cars_price_prediction_model_input['R_interior'] = model_input['interior_color'].apply(lambda x: get_RGB_component('R', x))
    used_cars_price_prediction_model_input['G_interior'] = model_input['interior_color'].apply(lambda x: get_RGB_component('G', x))
    used_cars_price_prediction_model_input['B_interior'] = model_input['interior_color'].apply(lambda x: get_RGB_component('B', x))

    used_cars_price_prediction_model_input['R_exterior'] = model_input['exterior_color'].apply(lambda x: get_RGB_component('R', x))
    used_cars_price_prediction_model_input['G_exterior'] = model_input['exterior_color'].apply(lambda x: get_RGB_component('G', x))
    used_cars_price_prediction_model_input['B_exterior'] = model_input['exterior_color'].apply(lambda x: get_RGB_component('B', x))

    used_cars_price_prediction_model_input['transmission_type_Automatic'] = model_input['transmission_type'].apply(lambda x: is_variable_value('Automatic', x))
    used_cars_price_prediction_model_input['transmission_type_Dual-clutch'] = model_input['transmission_type'].apply(lambda x: is_variable_value('Dual-clutch', x))
    used_cars_price_prediction_model_input['transmission_type_Manual'] = model_input['transmission_type'].apply(lambda x: is_variable_value('Manual', x))
    used_cars_price_prediction_model_input['transmission_type_Manumatic'] = model_input['transmission_type'].apply(lambda x: is_variable_value('Manumatic', x))
    used_cars_price_prediction_model_input['transmission_type_Variable'] = model_input['transmission_type'].apply(lambda x: is_variable_value('Variable', x))

    used_cars_price_prediction_model_input['drive_train_All-wheel Drive'] = model_input['drive_train'].apply(lambda x: is_variable_value('All-wheel Drive', x))
    used_cars_price_prediction_model_input['drive_train_Front-wheel Drive'] = model_input['drive_train'].apply(lambda x: is_variable_value('Front-wheel Drive', x))
    used_cars_price_prediction_model_input['drive_train_Rear-wheel Drive'] = model_input['drive_train'].apply(lambda x: is_variable_value('Rear-wheel Drive', x))

    used_cars_price_prediction_model_input['vehicle_type_Compact'] = model_input['vehicle_type'].apply(lambda x: is_variable_value('Compact', x))
    used_cars_price_prediction_model_input['vehicle_type_Large'] = model_input['vehicle_type'].apply(lambda x: is_variable_value('Large', x))
    used_cars_price_prediction_model_input['vehicle_type_Mid-size'] = model_input['vehicle_type'].apply(lambda x: is_variable_value('Mid-size', x))
    used_cars_price_prediction_model_input['vehicle_type_SUV'] = model_input['vehicle_type'].apply(lambda x: is_variable_value('SUV', x))
    used_cars_price_prediction_model_input['vehicle_type_Sport-car'] = model_input['vehicle_type'].apply(lambda x: is_variable_value('Sport-car', x))
    used_cars_price_prediction_model_input['vehicle_type_Subcompact'] = model_input['vehicle_type'].apply(lambda x: is_variable_value('Subcompact', x))
    used_cars_price_prediction_model_input['vehicle_type_Truck'] = model_input['vehicle_type'].apply(lambda x: is_variable_value('Truck', x))
    used_cars_price_prediction_model_input['vehicle_type_Van'] = model_input['vehicle_type'].apply(lambda x: is_variable_value('Van', x))

    used_cars_price_prediction_model_input['fuel_type_Diesel'] = model_input['fuel_type'].apply(lambda x: is_variable_value('Diesel', x))
    used_cars_price_prediction_model_input['fuel_type_Electric'] = model_input['fuel_type'].apply(lambda x: is_variable_value('Electric', x))
    used_cars_price_prediction_model_input['fuel_type_Gas'] = model_input['fuel_type'].apply(lambda x: is_variable_value('Gas', x))
    used_cars_price_prediction_model_input['fuel_type_Gasoline'] = model_input['fuel_type'].apply(lambda x: is_variable_value('Gasoline', x))
    used_cars_price_prediction_model_input['fuel_type_Hybrid'] = model_input['fuel_type'].apply(lambda x: is_variable_value('Hybrid', x))

    # Prediction of the price of an analogous used car
    model_input['predicted_used_car_price'] = used_cars_price_prediction_model(used_cars_price_prediction_model_input)

    # Discount the original price useful_life years to future
    model_input['residual_value'] = model_input['predicted_used_car_price']*pow(1 + i2 ,model_input['useful_life'])

    return model_input



residual_value_model_input = pd.DataFrame(columns=['useful_life', 'mileage', 'original_price', 'make', 'model', 'vehicle_type',
                                                     'fuel_type', 'mpg_city', 'mpg_highway', 'engine_displacement', 'engine_cylinders',
                                                     'transmission_type', 'num_speeds', 'drive_train', 'interior_color','exterior_color'])

car1 = {'useful_life': 6, 'mileage': 15000*6, 'original_price':55000, 'make':'audi', 'model': 'A5', 'vehicle_type': 'Mid-size',
        'fuel_type': 'Gasoline', 'mpg_city':0, 'mpg_highway':0, 'engine_displacement':2, 'engine_cylinders':4,
        'transmission_type':'Automatic', 'num_speeds':7, 'drive_train':'All-wheel Drive', 'interior_color':(0,0,0),'exterior_color':(0,0,0)}

car2 = {'useful_life': 4, 'mileage': 60000, 'original_price':37500, 'make':'mazda', 'model': 'CX-5', 'vehicle_type': 'SUV',
        'fuel_type': 'Hybrid', 'mpg_city':26, 'mpg_highway':30, 'engine_displacement':2.5, 'engine_cylinders':4,
        'transmission_type':'Automatic', 'num_speeds':6, 'drive_train':'All-wheel Drive', 'interior_color':(0,0,0),'exterior_color':(255,0,0)}   

residual_value_model_input = pd.DataFrame([car1, car2], columns=['useful_life', 'mileage', 'original_price', 'make', 'model', 'vehicle_type',
                                                     'fuel_type', 'mpg_city', 'mpg_highway', 'engine_displacement', 'engine_cylinders',
                                                     'transmission_type', 'num_speeds', 'drive_train', 'interior_color','exterior_color'])
residual_value_model_input = residual_value_model_input.astype({'useful_life': 'int', 'mileage': 'float', 'original_price': 'float',
         'make':'object', 'model': 'object', 'vehicle_type': 'object', 'fuel_type': 'object', 'mpg_city': 'float', 'mpg_highway': 'float',
         'engine_displacement': 'float', 'engine_cylinders':'float', 'transmission_type':'object', 'num_speeds':'float',
         'drive_train':'object'})

print('Residual value model input')         
display(residual_value_model_input)

residual_value_model_output = residual_value_prediction_model(residual_value_model_input, i1 = 0.03, i2 = 0.02)

print('Residual value model output')
display(residual_value_model_output)


Residual value model input


,useful_life,mileage,original_price,make,model,vehicle_type,fuel_type,mpg_city,mpg_highway,engine_displacement,engine_cylinders,transmission_type,num_speeds,drive_train,interior_color,exterior_color
0,6,90000.0,55000.0,audi,A5,Mid-size,Gasoline,0.0,0.0,2.0,4.0,Automatic,7.0,All-wheel Drive,"(0, 0, 0)","(0, 0, 0)"
1,4,60000.0,37500.0,mazda,CX-5,SUV,Hybrid,26.0,30.0,2.5,4.0,Automatic,6.0,All-wheel Drive,"(0, 0, 0)","(255, 0, 0)"


Residual value model output


,useful_life,mileage,original_price,make,model,vehicle_type,fuel_type,mpg_city,mpg_highway,engine_displacement,engine_cylinders,transmission_type,num_speeds,drive_train,interior_color,exterior_color,former_original_price,predicted_used_car_price,residual_value
0,6,90000.0,55000.0,audi,A5,Mid-size,Gasoline,0.0,0.0,2.0,4.0,Automatic,7.0,All-wheel Drive,"(0, 0, 0)","(0, 0, 0)",46061.634118,21908.998047,24673.090244
1,4,60000.0,37500.0,mazda,CX-5,SUV,Hybrid,26.0,30.0,2.5,4.0,Automatic,6.0,All-wheel Drive,"(0, 0, 0)","(255, 0, 0)",33318.264297,24376.785156,26386.216211
